In [6]:
# =====================================================
# PERTEMUAN 10 - RANDOM FOREST CUSTOMER CHURN
# =====================================================

# ==========================
# Import Library
# ==========================
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score
)

# ==========================
# Langkah 1 : Load Dataset
# ==========================
df = pd.read_csv("telco_churn.csv")

print("Ukuran Dataset:")
print(df.shape)

print("\n5 Data Pertama")
print(df.head())

print("\nTipe Data")
print(df.dtypes)

print("\nProporsi Kelas Churn")
print(df["Churn"].value_counts(normalize=True))

Ukuran Dataset:
(7043, 21)

5 Data Pertama
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport Strea

In [7]:
# Hilangkan customerID karena bukan fitur prediksi
if "customerID" in df.columns:
    df.drop("customerID", axis=1, inplace=True)

# Mengubah TotalCharges menjadi numerik
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

# Encoding target
df["Churn"] = df["Churn"].map({
    "Yes":1,
    "No":0
})

# One Hot Encoding
df = pd.get_dummies(df, drop_first=True)

# Pisahkan fitur dan target
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Train Test Split
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training :", X_tr.shape)
print("Testing  :", X_te.shape)

Training : (5634, 30)
Testing  : (1409, 30)


/tmp/ipykernel_1309/714825329.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


In [8]:
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [9]:
# Prediksi kelas
pred = rf.predict(X_te)

# Prediksi probabilitas
proba = rf.predict_proba(X_te)[:,1]

print("Classification Report")
print(classification_report(y_te, pred))

print("ROC-AUC :", roc_auc_score(y_te, proba))

Classification Report
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC : 0.8246208891988943


In [10]:
hasil = pd.DataFrame({
    "Actual": y_te.values,
    "Prediksi": pred,
    "Probabilitas Churn": proba
})

print(hasil.head(10))

   Actual  Prediksi  Probabilitas Churn
0       0         0            0.000000
1       0         1            0.786667
2       0         0            0.090000
3       0         0            0.280000
4       0         0            0.000000
5       0         0            0.416667
6       0         0            0.393333
7       0         0            0.110000
8       0         0            0.006667
9       1         0            0.460000


**Kesimpulan**

Dari praktikum ini, saya belajar bahwa Random Forest dapat digunakan untuk memprediksi pelanggan yang berpotensi melakukan churn. Saya juga memahami bahwa pada dataset yang tidak seimbang, metrik seperti Recall dan F1-Score lebih tepat digunakan dibandingkan hanya melihat akurasi. Hasil prediksi probabilitas dari model dapat membantu perusahaan menentukan pelanggan yang perlu diprioritaskan untuk dipertahankan.
